# Anomaly Injection - Experimentation & Building the Anomaly Injection Pipeline

In this notebook, we will create a few controlled anomaly cases from the clean AWS dataset.

The original clean dataset will not be modified.

In [1]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


# Find the main project folder
current_path = Path.cwd()

project_root = None

for path in [current_path] + list(current_path.parents):
    if (path / "01_ml").exists():
        project_root = path
        break



if project_root is None:
    raise FileNotFoundError("Could not find the Skyguard_AI project folder.")


print("Project root:", project_root)

Project root: c:\Users\sanch\Documents\Git_hub\Skyguard_AI


In [2]:
clean_data_path = (
    project_root
    / "01_ml"
    / "01_data"
    / "02_processed"
    / "aws_stations_with_synthetic_humidity_clean.csv"
)

df = pd.read_csv(clean_data_path)

df["date_of_record"] = pd.to_datetime(df["date_of_record"])

# Keep the data in station-wise chronological order
df = df.sort_values(
    ["station_name", "date_of_record"]
).reset_index(drop=True)

print("Clean dataset loaded.")
print("Shape:", df.shape)

Clean dataset loaded.
Shape: (20888, 15)


In [3]:
df["date_of_record"] = pd.to_datetime(df["date_of_record"])

df = df.sort_values(
    ["station_name", "date_of_record"]
).reset_index(drop=True)

clean_df = df.copy()

print("Clean baseline copied.")
print("Rows:", len(clean_df))

Clean baseline copied.
Rows: 20888


### 1. Ground-truth columns

- is_injected       = True
- anomaly_category  = SENSOR_FAULT
- anomaly_type      = SUDDEN_SPIKE
- affected_feature  = avg_temp
- original_value    = 25.4
- modified_value    = 42.7
- anomaly_id        = SF_0001

In [4]:
df["is_injected"] = False
df["anomaly_category"] = "NORMAL"
df["anomaly_type"] = None
df["affected_feature"] = None
df["original_value"] = np.nan
df["modified_value"] = np.nan
df["anomaly_id"] = None

### 2. Anomaly log

In [5]:
anomaly_log = []

print("Ground-truth tracking is ready.")

Ground-truth tracking is ready.


### 3. Choosing a deterministic row

In [6]:
station_name = "Mandi"

station_rows = df[
    df["station_name"] == station_name
].copy()

station_rows = station_rows.sort_values(
    "date_of_record"
).reset_index()

print("Available rows for", station_name, ":", len(station_rows))

Available rows for Mandi : 1489


### 4. Inject sudden spike

In [7]:
target_index = station_rows.loc[500, "index"]

old_value = df.loc[target_index, "avg_temp"]

new_value = old_value + 15.0

df.loc[target_index, "avg_temp"] = new_value

df.loc[target_index, "is_injected"] = True
df.loc[target_index, "anomaly_category"] = "SENSOR_FAULT"
df.loc[target_index, "anomaly_type"] = "SUDDEN_SPIKE"
df.loc[target_index, "affected_feature"] = "avg_temp"
df.loc[target_index, "original_value"] = old_value
df.loc[target_index, "modified_value"] = new_value
df.loc[target_index, "anomaly_id"] = "SF_0001"

anomaly_log.append({
    "anomaly_id": "SF_0001",
    "station_name": station_name,
    "anomaly_category": "SENSOR_FAULT",
    "anomaly_type": "SUDDEN_SPIKE",
    "affected_feature": "avg_temp",
    "original_value": old_value,
    "modified_value": new_value
})

print("Sudden spike injected.")
print("Original value:", old_value)
print("New value     :", new_value)

Sudden spike injected.
Original value: 27.2
New value     : 42.2


### 5. Verifying the spike

In [8]:
df.loc[
    target_index,
    [
        "station_name",
        "date_of_record",
        "avg_temp",
        "is_injected",
        "anomaly_category",
        "anomaly_type",
        "affected_feature",
        "original_value",
        "modified_value"
    ]
]

station_name                      Mandi
date_of_record      2022-05-17 00:00:00
avg_temp                           42.2
is_injected                        True
anomaly_category           SENSOR_FAULT
anomaly_type               SUDDEN_SPIKE
affected_feature               avg_temp
original_value                     27.2
modified_value                     42.2
Name: 10958, dtype: object

In [9]:
anomaly_log

[{'anomaly_id': 'SF_0001',
  'station_name': 'Mandi',
  'anomaly_category': 'SENSOR_FAULT',
  'anomaly_type': 'SUDDEN_SPIKE',
  'affected_feature': 'avg_temp',
  'original_value': np.float64(27.2),
  'modified_value': np.float64(42.2)}]

In [10]:
df.shape

(20888, 22)

---

###  II. Let's simulate a sensor getting stuck at one value                                                                          - 02/09/26

In [11]:

station_name = "Mandi"

station_rows = df[
    df["station_name"] == station_name
].sort_values("date_of_record")

start_position = 700
fault_length = 5

target_rows = station_rows.iloc[
    start_position:start_position + fault_length
]

stuck_value = target_rows.iloc[0]["relative_humidity"]

print("Stuck value:", stuck_value)
print("Affected dates:")
print(target_rows["date_of_record"].to_list())

Stuck value: 81.4229542961873
Affected dates:
[Timestamp('2022-12-05 00:00:00'), Timestamp('2022-12-06 00:00:00'), Timestamp('2022-12-07 00:00:00'), Timestamp('2022-12-08 00:00:00'), Timestamp('2022-12-09 00:00:00')]


In [12]:
# Save the original humidity values before changing them
original_values = target_rows["relative_humidity"].copy()

# Make the humidity sensor stay at one fixed value
df.loc[target_rows.index, "relative_humidity"] = stuck_value

# Add ground-truth information
df.loc[target_rows.index, "is_injected"] = True
df.loc[target_rows.index, "anomaly_category"] = "SENSOR_FAULT"
df.loc[target_rows.index, "anomaly_type"] = "STUCK_AT"
df.loc[target_rows.index, "affected_feature"] = "relative_humidity"
df.loc[target_rows.index, "original_value"] = original_values.values
df.loc[target_rows.index, "modified_value"] = stuck_value

for i, row_index in enumerate(target_rows.index):
    df.loc[row_index, "anomaly_id"] = f"SF_STUCK_{i + 1:03d}"

print("Stuck-at fault injected successfully.")

Stuck-at fault injected successfully.


### II a. Verify before vs after

In [13]:
comparison = pd.DataFrame({
    "date": target_rows["date_of_record"].values,
    "original_relative_humidity": original_values.values,
    "modified_relative_humidity": df.loc[target_rows.index, "relative_humidity"].values
})

comparison

,date,original_relative_humidity,modified_relative_humidity
0,2022-12-05,81.422954,81.422954
1,2022-12-06,83.768313,81.422954
2,2022-12-07,82.016944,81.422954
3,2022-12-08,83.419082,81.422954
4,2022-12-09,80.630600,81.422954


### II b. Check ground truth

In [14]:
df.loc[
    target_rows.index,
    [
        "station_name",
        "date_of_record",
        "relative_humidity",
        "is_injected",
        "anomaly_category",
        "anomaly_type",
        "affected_feature",
        "original_value",
        "modified_value",
        "anomaly_id"
    ]
]

,station_name,date_of_record,relative_humidity,is_injected,anomaly_category,anomaly_type,affected_feature,original_value,modified_value,anomaly_id
11158,Mandi,2022-12-05,81.422954,True,SENSOR_FAULT,STUCK_AT,relative_humidity,81.422954,81.422954,SF_STUCK_001
11159,Mandi,2022-12-06,81.422954,True,SENSOR_FAULT,STUCK_AT,relative_humidity,83.768313,81.422954,SF_STUCK_002
11160,Mandi,2022-12-07,81.422954,True,SENSOR_FAULT,STUCK_AT,relative_humidity,82.016944,81.422954,SF_STUCK_003
11161,Mandi,2022-12-08,81.422954,True,SENSOR_FAULT,STUCK_AT,relative_humidity,83.419082,81.422954,SF_STUCK_004
11162,Mandi,2022-12-09,81.422954,True,SENSOR_FAULT,STUCK_AT,relative_humidity,80.630600,81.422954,SF_STUCK_005


#### Ek VERY important point bhai

Yahan humne 5 rows ko 5 separate anomalies nahi maana conceptually.
Ye actually one stuck-at event hai jo 5 observations tak chala.
Abhi humne demonstration ke liye IDs per-row rakhi hain, but final injector banate waqt hum isko better structure denge:

event_id = SF_STUCK_001
start = 2022-12-05
end   = 2022-12-09

So later model evaluation mein hum event-level detection bhi kar sakte hain.

---

### III Let's simulate one missing observation from Mandi

In [15]:
station_name = "Mandi"

station_rows = df[
    df["station_name"] == station_name
].sort_values("date_of_record")

target_row = station_rows.iloc[900]

target_index = target_row.name

missing_date = target_row["date_of_record"]

print("Station :", station_name)
print("Missing date:", missing_date)
print("Original row index:", target_index)

Station : Mandi
Missing date: 2023-06-23 00:00:00
Original row index: 11358


### III a. Record the missing event

In [16]:
# Save the information before removing the record
missing_record = df.loc[target_index].copy()

anomaly_log.append({
    "anomaly_id": "TG_MISSING_001",
    "station_name": station_name,
    "date": missing_date,
    "anomaly_category": "TRANSMISSION_GLITCH",
    "anomaly_type": "MISSING_RECORD",
    "affected_feature": "entire_record"
})

print("Missing-record event has been logged.")

Missing-record event has been logged.


### IV Actually remove the record

In [17]:
# Remove the record from the experimental dataset
df = df.drop(index=target_index).reset_index(drop=True)

print("Record removed from the experimental dataset.")
print("New shape:", df.shape)

Record removed from the experimental dataset.
New shape: (20887, 22)


### IV a. Verify that a gap now exists

In [18]:
# Look at the dates around the removed observation
mandi_dates = df[
    df["station_name"] == station_name
]["date_of_record"].sort_values()

nearby_dates = mandi_dates[
    (mandi_dates >= missing_date - pd.Timedelta(days=2))
    & (mandi_dates <= missing_date + pd.Timedelta(days=2))
]

print(nearby_dates.to_list())

[Timestamp('2023-06-21 00:00:00'), Timestamp('2023-06-22 00:00:00'), Timestamp('2023-06-24 00:00:00'), Timestamp('2023-06-25 00:00:00')]


## Verifying is clean baseline  still intact

In [19]:
clean_row_exists = (
    clean_df["station_name"].eq(station_name)
    & clean_df["date_of_record"].eq(missing_date)
).any()

print("Original record still exists in clean_df:", clean_row_exists)

Original record still exists in clean_df: True


---

## Testing Reusable Anomaly Functions

In [20]:
import sys

anomaly_path = project_root / "01_ml" / "03_src" / "02_anomaly"
sys.path.insert(0, str(anomaly_path))

from sensor_faults import add_sudden_spike, add_stuck_at
from transmission_faults import remove_record

print("Anomaly functions imported successfully!")

Anomaly functions imported successfully!


#### Test sudden spike

In [21]:
test_df = clean_df.copy()

result = add_sudden_spike(
    test_df,
    row_index=5000,
    column="avg_temp",
    increase=15.0
)

print("Original value:", result["original_value"])
print("Modified value:", result["modified_value"])

Original value: 34.2
Modified value: 49.2


#### Testing Stuck at particular value

In [22]:
test_df = clean_df.copy()

target_rows = test_df[
    test_df["station_name"] == "Mandi"
].sort_values("date_of_record").iloc[700:705].index

result = add_stuck_at(
    test_df,
    target_rows,
    "relative_humidity"
)

print("Original values:")
print(result["original_values"])

print("\nStuck value:")
print(result["stuck_value"])

print("\nModified values:")
test_df.loc[
    target_rows,
    "relative_humidity"
].tolist()

Original values:
[81.4229542961873, 83.76831302052653, 82.01694359015897, 83.41908206071882, 80.63060024754807]

Stuck value:
81.4229542961873

Modified values:


[81.4229542961873,
 81.4229542961873,
 81.4229542961873,
 81.4229542961873,
 81.4229542961873]

#### Test missing record

In [23]:
test_df = clean_df.copy()

target_index = test_df[
    (test_df["station_name"] == "Mandi")
].sort_values("date_of_record").iloc[900].name

test_df, removed_record = remove_record(
    test_df,
    target_index
)

print("Removed station:", removed_record["station_name"])
print("Removed date:", removed_record["date_of_record"])
print("New dataset shape:", test_df.shape)

Removed station: Mandi
Removed date: 2023-06-23 00:00:00
New dataset shape: (20887, 15)


Why we're doing this before the full injector

This step might look boring but it's an essential part of ML workflow samjheeee....

We're validating:

              sensor_faults.py
                  ↓
            works independently 

            transmission_faults.py
                  ↓
            works independently 

THEN

            anomaly_injector.py
                  ↓
            orchestrates everything

---

## Testing Combined Anomaly Injector

In [24]:
from anomaly_injector import inject_test_anomalies

test_df, test_anomaly_log = inject_test_anomalies(clean_df)

print("Original shape:", clean_df.shape)
print("Modified shape:", test_df.shape)

print("\nAnomaly log:")
display(test_anomaly_log)

Original shape: (20888, 15)
Modified shape: (20887, 15)

Anomaly log:


,anomaly_id,station_name,anomaly_category,anomaly_type,affected_feature,start_date,end_date,severity,original_value,modified_value,original_values,affected_rows
0,TEST_SF_001,Mandi,SENSOR_FAULT,SUDDEN_SPIKE,avg_temp,2022-05-17,2022-05-17,high,27.2,42.200000,NaN,NaN
1,TEST_SF_002,Mandi,SENSOR_FAULT,STUCK_AT,relative_humidity,2022-12-05,2022-12-09,medium,NaN,81.422954,"[81.4229542961873, 83.76831302052653, 82.01694...",5.0
2,TEST_TG_001,Mandi,TRANSMISSION_GLITCH,MISSING_RECORD,entire_record,2023-06-23,2023-06-23,medium,NaN,NaN,NaN,NaN


In [25]:
print("Clean dataset rows:", len(clean_df))
print("Test dataset rows :", len(test_df))

print(
    "\nMissing record date:",
    test_anomaly_log.loc[
        test_anomaly_log["anomaly_type"] == "MISSING_RECORD",
        "start_date"
    ].iloc[0]
)

Clean dataset rows: 20888
Test dataset rows : 20887

Missing record date: 2023-06-23 00:00:00


In [26]:
assert len(clean_df) == 20888
assert len(test_df) == 20887

print("\nClean dataset is still untouched.")


Clean dataset is still untouched.


---

### Anomaly_injector.py - event-level design shift

In [27]:
import sys

anomaly_path = project_root / "01_ml" / "03_src" / "02_anomaly"

if str(anomaly_path) not in sys.path:
    sys.path.insert(0, str(anomaly_path))

In [28]:
from anomaly_injector import create_anomaly_event

print("create_anomaly_event imported successfully!")

create_anomaly_event imported successfully!


In [29]:
from anomaly_injector import create_anomaly_event

event = create_anomaly_event(
    anomaly_id="TEST_SF_002",
    station_name="Mandi",
    anomaly_category="SENSOR_FAULT",
    anomaly_type="STUCK_AT",
    affected_feature="humidity",
    start_date="2022-12-05",
    end_date="2022-12-09",
    severity="medium"
)

event

{'anomaly_id': 'TEST_SF_002',
 'station_name': 'Mandi',
 'anomaly_category': 'SENSOR_FAULT',
 'anomaly_type': 'STUCK_AT',
 'affected_feature': 'humidity',
 'start_date': '2022-12-05',
 'end_date': '2022-12-09',
 'severity': 'medium'}

---

### Before final injection __ Helper check 

In [30]:
from anomaly_injector import select_station_row

test_index = select_station_row(
    clean_df,
    station_name="Mandi",
    position=500
)

print("Selected row index:", test_index)
print(clean_df.loc[
    test_index,
    ["station_name", "date_of_record", "avg_temp", "relative_humidity"]
])

Selected row index: 10958
station_name                       Mandi
date_of_record       2022-05-17 00:00:00
avg_temp                            27.2
relative_humidity              63.892738
Name: 10958, dtype: object


In [31]:
index_100 = select_station_row(
    clean_df,
    station_name="Mandi",
    position=100
)

index_500 = select_station_row(
    clean_df,
    station_name="Mandi",
    position=500
)

print("100th observation:")
print(clean_df.loc[index_100, ["date_of_record", "avg_temp"]])

print("\n500th observation:")
print(clean_df.loc[index_500, ["date_of_record", "avg_temp", "relative_humidity"]])

100th observation:
date_of_record    2021-04-12 00:00:00
avg_temp                         21.5
Name: 10558, dtype: object

500th observation:
date_of_record       2022-05-17 00:00:00
avg_temp                            27.2
relative_humidity              63.892738
Name: 10958, dtype: object
